In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sys
import os
sys.path.append(os.getcwd())

import matplotlib as mpl
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["TeX Gyre Pagella", "Book Antiqua", "Palatino Linotype", "DejaVu Serif"]
})

import pickle
import jax
import jax.numpy as jnp

from conditional_diffusion_3d_model import ConditionalUNet3D, DiffusionModelConfig
from train_conditional_diffusion import preload_hdf5_to_memory, train_model

### Import data

In [ ]:
val_path = "/projects/mccleary_group/habjan.e/TNG/Data/conditional_diffusion_data/cond_diffusion_test.h5"
data_dict = preload_hdf5_to_memory(val_path)

sample_path = '/projects/mccleary_group/habjan.e/TNG/Data/conditional_diffusion_data/cd_samples/sampled_cubes_validx_0000.npz'
data = np.load(sample_path)
data_im, data_true_cube, data_sampled_cubes, seeds = data['conditioning_images'], data['true_cube'], data['sampled_cubes'], data['seeds']

### Plot data

In [ ]:
imgs = data_im
cmap = 'cubehelix'

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(15, 10), constrained_layout=True)
fig.set_constrained_layout_pads(w_pad=0.2, h_pad=0.2, wspace=0.1, hspace=0.1)
ax1, ax2, ax3 = axes[0], axes[1], axes[2]

im1 = ax1.imshow(imgs[0], vmin = np.quantile(imgs[0], 0.5), vmax = np.quantile(imgs[0], 0.98), cmap=cmap, origin="upper")
ax1.set_xlabel("X-coordinate", fontsize=14, fontweight='semibold')
ax1.set_ylabel("Y-coordinate", fontsize=14, fontweight='semibold')
cbar = fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.02)
cbar.set_label("Projected Mass", fontsize=14, fontweight='semibold')

im2 = ax2.imshow(imgs[1], cmap=cmap, origin="upper")
ax2.set_xlabel("X-coordinate", fontsize=14, fontweight='semibold')
ax2.set_ylabel("Y-coordinate", fontsize=14, fontweight='semibold')
cbar = fig.colorbar(im2, ax=ax2, fraction=0.046, pad=0.02)
cbar.set_label("Galaxy Density", fontsize=14, fontweight='semibold')

im3 = ax3.imshow(imgs[2], cmap=cmap, origin="upper")
ax3.set_xlabel("X-coordinate", fontsize=14, fontweight='semibold')
ax3.set_ylabel("Y-coordinate", fontsize=14, fontweight='semibold')
cbar = fig.colorbar(im3, ax=ax3, fraction=0.046, pad=0.02)
cbar.set_label("$\mathbf{V_{z}}$", fontsize=14, fontweight='semibold')

#fig.savefig("/home/habjan.e/TNG/cluster_deprojection/figures/cnnmlp_data.png", bbox_inches="tight")
plt.show()

### Projected true density cube

In [ ]:
cube_mean = data_dict["metadata"]["cube_log10_mean"]
cube_std  = data_dict["metadata"]["cube_log10_std"]

log10_rho = data_true_cube * cube_std + cube_mean
rho = 10**log10_rho

rho_im = np.sum(rho, axis = 1)

imgs = (np.log10(rho_im ) - data_dict["metadata"]["proj_mass_log10_mean"]) / data_dict["metadata"]["proj_mass_log10_std"]
cmap = 'cubehelix'

fig, ax1 = plt.subplots(nrows=1, ncols=1, figsize=(5, 5), constrained_layout=False)
fig.set_constrained_layout_pads(w_pad=0.1, h_pad=0.2, wspace=-0.1, hspace=0.1)

im1 = ax1.imshow(imgs, vmin = np.quantile(imgs, 0.5), vmax = np.quantile(imgs, 0.98), cmap=cmap, origin="upper")
ax1.set_xlabel("Z-coordinate", fontsize=14, fontweight='semibold')
ax1.set_ylabel("X-coordinate", fontsize=14, fontweight='semibold')
cbar = fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.02)
cbar.set_label("Projected Mass", fontsize=14, fontweight='semibold')

### Project the density cube - it should look just like the projected cube above

In [ ]:
cube_mean = data_dict["metadata"]["cube_log10_mean"]
cube_std  = data_dict["metadata"]["cube_log10_std"]

log10_rho = data_sampled_cubes * cube_std + cube_mean
rho_samp = 10**log10_rho

rho_im = np.sum(rho_samp, axis = 2)

samp_imgs = (np.log10(rho_im ) - data_dict["metadata"]["proj_mass_log10_mean"]) / data_dict["metadata"]["proj_mass_log10_std"]
cmap = 'cubehelix'
fs = 12

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(13, 10), constrained_layout=False)
fig.set_constrained_layout_pads(w_pad=0.1, h_pad=0.2, wspace=-0.1, hspace=0.1)
ax1, ax2, ax3, ax4 = axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]

ind = 0
im1 = ax1.imshow(samp_imgs[ind], vmin = np.quantile(samp_imgs[ind], 0.5), vmax = np.quantile(samp_imgs[ind], 0.98), cmap=cmap, origin="upper")
ax1.set_xlabel("Z-coordinate", fontsize=14, fontweight='semibold')
ax1.set_ylabel("X-coordinate", fontsize=14, fontweight='semibold')
ax1.text(0.025, 0.925, f'Seed: {seeds[ind]}', transform=ax1.transAxes, color='white', weight='semibold', size=fs)
cbar = fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.02)
cbar.set_label("Projected Mass", fontsize=14, fontweight='semibold')

ind = 1
im2 = ax2.imshow(samp_imgs[ind], vmin = np.quantile(samp_imgs[ind], 0.5), vmax = np.quantile(samp_imgs[ind], 0.98), cmap=cmap, origin="upper")
ax2.set_xlabel("Z-coordinate", fontsize=14, fontweight='semibold')
ax2.set_ylabel("X-coordinate", fontsize=14, fontweight='semibold')
ax2.text(0.025, 0.925, f'Seed: {seeds[ind]}', transform=ax2.transAxes, color='white', weight='semibold', size=fs)
cbar = fig.colorbar(im2, ax=ax2, fraction=0.046, pad=0.02)
cbar.set_label("Projected Mass", fontsize=14, fontweight='semibold')

ind = 2
im3 = ax3.imshow(samp_imgs[ind], vmin = np.quantile(samp_imgs[ind], 0.5), vmax = np.quantile(samp_imgs[ind], 0.98), cmap=cmap, origin="upper")
ax3.set_xlabel("Z-coordinate", fontsize=14, fontweight='semibold')
ax3.set_ylabel("X-coordinate", fontsize=14, fontweight='semibold')
ax3.text(0.025, 0.925, f'Seed: {seeds[ind]}', transform=ax3.transAxes, color='white', weight='semibold', size=fs)
cbar = fig.colorbar(im3, ax=ax3, fraction=0.046, pad=0.02)
cbar.set_label("Projected Mass", fontsize=14, fontweight='semibold')

ind = 3
im4 = ax4.imshow(samp_imgs[ind], vmin = np.quantile(samp_imgs[ind], 0.5), vmax = np.quantile(samp_imgs[ind], 0.98), cmap=cmap, origin="upper")
ax4.set_xlabel("Z-coordinate", fontsize=14, fontweight='semibold')
ax4.set_ylabel("X-coordinate", fontsize=14, fontweight='semibold')
ax4.text(0.025, 0.925, f'Seed: {seeds[ind]}', transform=ax4.transAxes, color='white', weight='semibold', size=fs)
cbar = fig.colorbar(im4, ax=ax4, fraction=0.046, pad=0.02)
cbar.set_label("Projected Mass", fontsize=14, fontweight='semibold')

### Look at coorelation between different density maps

In [ ]:
nonzero_pix = data_true_cube > -5

rho_true_flat = np.log10(rho[nonzero_pix])
rho_samp_flat = np.log10(np.array([rho_samp[i][nonzero_pix] for i in range(rho_samp.shape[0])]))

one_one = np.linspace(-5, 20, 100)

x_plot, y_plot = rho_true_flat, rho_samp_flat

colors = np.array(['red', 'green', 'blue', 'orange'])

for i in range(rho_samp.shape[0]):

    fig, axs = plt.subplots(1, 1, figsize=(6, 5), gridspec_kw={'wspace': 0.35})

    axs.scatter(x_plot, y_plot[i], c=colors[i], label=f'Seed: {seeds[i]}', s=10)

    axs.plot(one_one, one_one, c='k', linestyle='--')
    axs.set_xlabel(r'BAHAMAS Mass-Density $\left[ log_{10}(M_{\odot} Mpc^{-3}) \right]$', fontsize = 15)
    axs.set_ylabel(r'Predicted $\left[ log_{10}(M_{\odot} Mpc^{-3}) \right]$', fontsize = 15)
    lims = np.concatenate([x_plot, y_plot.flatten()])
    axs.set_xlim(10, 17)
    axs.set_ylim(10, 17)
    axs.legend()

    plt.show()